In [13]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali


#fit 
def Malus(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return A*np.cos(t - C)**2 + B

def modellopy(t, A, B, C):         
    return A*np.cos(t - C)**2 + B


def Malus_deriv(x, par):
    t = x
    dt = 1e-6
    return ((modellopy(t+dt/2, par[0], par[1], par[2])-modellopy(t-dt/2, par[0], par[1], par[2]))/dt)
    


xlsx ="Esperienza4.xlsx"
df = pd.read_excel(xlsx, sheet_name= "Dati")
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.to_numpy(dtype = 'float64')

x = df[:, 1]     
y = df[:, 2]     

err_x = df[:, 3]     
err_y = df[:, 4]   

'''
index = [10,11]

x = np.delete(x, index)
y = np.delete(y, index)
err_x = np.delete(err_x, index)
err_y = np.delete(err_y, index)
'''

#minimizzazzione con ROOT
MalusFit = ROOT.TF1('model', Malus, min(x), max(x), 3)
MalusFit.SetParameters(y.max() - y.min(), y.min(), x[np.argmax(y)])     # A è Vmax-Vmin, B è Vmin, C è posizione del massimo
MalusFit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(MalusFit, "SR") # SR significa che voglio fare il fit senza disegnare la funzione e senza stampare a video i risultati del fit 

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = MalusFit.GetParameters()

errors = MalusFit.GetParErrors()

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((Malus_deriv(x, par)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(MalusFit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("Immagini/malus_fit.png")
ROOT.gStyle.SetOptFit(0000)    

Fit status: 0
Parametri stimati:
A = 3.030382200856419 ± 0.0011678972026484317
B = 3.4438182468705816 ± 0.0008058079090724105
C = 0.029335876643752663 ± 0.0004149904412122199
Chi2 = 51767.73482800658, ndf = 178
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      51767.7
NDf                       =          178
Edm                       =   2.3493e-08
NCalls                    =           85
A                         =      3.03038   +/-   0.0011679   
B                         =      3.44382   +/-   0.000805808 
C                         =    0.0293359   +/-   0.00041499  


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/malus_fit.png has been created


In [14]:
if np.abs(par[2]) < 3*errors[2]:
        print("C è compatibile con 0")
else :
        print("C NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*errors[1]:
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


C NON è compatibile con 0
---
B NON è compatibile con 0


In [15]:

y_attesi = np.array([MalusFit.Eval(x_val) for x_val in x])
residuals_norm = (y - y_attesi) / err_yeff


gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm), max(residuals_norm), 3)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
for val in residuals_norm:
    hist.Fill(val)

gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
    
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()

ROOT.gStyle.SetOptFit(1111)  
c2.Draw()
c2.SaveAs("Immagini/malus_hist.png")


****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      33.7434
NDf                       =           16
Edm                       =  7.17924e-07
NCalls                    =           69
const                     =      16.5086   +/-   1.82715     
mu                        =       5.7087   +/-   1.44541     
sigma                     =      15.9176   +/-   1.24402      	 (limited)
Fit status: 0
Parametri stimati:
const = 16.50860572177809 ± 1.8271458840230146
#mu = 5.708700630921501 ± 1.4454130915810488
#sigma = 15.917592504359918 ± 1.244018847698519
Chi2 = 33.74343424773684


Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file Immagini/malus_hist.png has been created


In [16]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")
if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [17]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = y - y_attesi

line_fit = ROOT.TF1('line', line, min(x), max(x), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(x), x, residuals, np.zeros_like(residuals), err_yeff)
gr2.SetTitle("Residui in funzione di #theta; #theta [rad]; V - V_{att} [V]")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("Immagini/malus_residui.png")

Fit status: 0
Parametri stimati:
m = -0.041829723482273595 ± 0.00027899730971285845
q = 0.12720747780864539 ± 0.0010012040955393287
Chi2 = 29289.098487300867
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      29289.1
NDf                       =          179
Edm                       =   7.1491e-18
NCalls                    =           38
m                         =   -0.0418297   +/-   0.000278997 
q                         =     0.127207   +/-   0.0010012   


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file Immagini/malus_residui.png has been created


In [18]:
# stima    disfunzionalita polarizzatore
Vmax = MalusFit.GetParameter(0) + MalusFit.GetParameter(1)     # A + B
dV_semi = (residuals.max() - residuals.min())/2

print(f"Vmax = {Vmax:.3f} V")
print(f"DeltaV (semiampiezza) = {dV_semi:.3f} V   ->  DeltaV/Vmax = {dV_semi/Vmax:.3f}")
print(f"DeltaV (picco-picco)  = {2*dV_semi:.3f} V   ->  DeltaV/Vmax = {2*dV_semi/Vmax:.3f}")


Vmax = 6.474 V
DeltaV (semiampiezza) = 0.296 V   ->  DeltaV/Vmax = 0.046
DeltaV (picco-picco)  = 0.591 V   ->  DeltaV/Vmax = 0.091
